In [1]:
import os
import pandas as pd
import dask.dataframe as dd
from tqdm.notebook import tqdm
import time
import datetime

from utilities import get_date, get_subzone_name, walk_through_files, get_subzone_name_from_dir

import warnings
warnings.filterwarnings('ignore')

from dask.distributed import LocalCluster, Client
cluster = LocalCluster(n_workers=2)
client = Client(cluster)  # Connect to distributed cluster and override default
client

Client Scheduler: tcp://127.0.0.1:53539 Dashboard: http://127.0.0.1:53538/status,Cluster Workers: 2 Cores: 12 Memory: 34.29 GB


In [2]:
RECORDS_PATH =  os.path.join('..', 'raw-files', 'sg_subzones_devices')
SAVEPATH = os.path.join('intermediate-files', 'device-area-maxcount.csv.gz')
SUBZONE_PATH = os.path.join('intermediate-files', 'citydata-subzones.csv')
ALTERNATE_DUMP_PATH = os.path.join('..', '..', '..', 'wgw-lf', 'datadump')
DEVICE_ID_DICT_PATH = os.path.join('intermediate-files', 'deviceID-list', 'deviceID-list-chunk*.csv.gz')

#### Replicate datadump with (integer) IDs

In [15]:
if True:
    
    # Get device ID dict DEVICE_ID_DICT_PATH
    devices = (dd.read_csv(DEVICE_ID_DICT_PATH, compression='gzip') ).compute()
    devices = dict(zip(devices['hashid'], devices['xid']))

    
    # Get area ID dict
    subzones = pd.read_csv(SUBZONE_PATH)
    subzones = dict(zip(subzones['subzone'].str.lower(), subzones['areaid']))

    
    file_lists = tuple(walk_through_files(RECORDS_PATH, 'csv.gz', returnboth=True))

    
    # Create alternate datadump
    for (filepath, filename) in tqdm(file_lists):
        # open file
        _date = get_date(filename)
        # https://stackoverflow.com/questions/19801727/convert-datetime-to-unix-timestamp-and-convert-it-back-in-python
        _date = time.mktime(_date.timetuple()) 
        _subzone = get_subzone_name(filename)

        df = (pd.read_csv(os.path.join(filepath, filename), compression='gzip', names=['device', 'ios'])
              .assign(date=_date)
              .assign(area_id=subzones[_subzone])
    #           .assign(device=lambda x: [devices[device] for device in x['device']])
              .assign(device=lambda x: [devices[device] for device in x['device']])
              .rename(columns={'device': 'device_xid'})
              .astype({'area_id':'int', 'device_xid': 'int', 'ios': 'int', 'date': 'int'})
             ).to_csv(os.path.join(ALTERNATE_DUMP_PATH, filename), index=False, compression='gzip')  

#### Get max count

In [4]:
file_lists = tuple(walk_through_files(ALTERNATE_DUMP_PATH, 'csv.gz', fullpath=True))

In [5]:
ddf = (dd.read_csv(file_lists, compression='gzip', usecols=['device_xid', 'area_id'], blocksize=None)
       .groupby(['device_xid', 'area_id']).size()
       .reset_index()
       .rename(columns={0: 'count'})
       .astype({'count': 'int', 'area_id': 'int', 'device_xid': 'int'})
       .loc[lambda x: x.groupby('device_xid')['count'].transform(max) == x['count']]
      )

In [6]:
ddf

,device_xid,area_id,count
npartitions=1,,,
,int32,int32,int32
,...,...,...


In [7]:
df = ddf.compute()

In [8]:
df.head()

,device_xid,area_id,count
0,133438,0,89
1,260901,0,12
2,440462,0,86
5,957832,0,1
7,1278377,0,40


In [9]:
df.to_csv(SAVEPATH, index=False, compression='gzip')

#### Testing output using dummyfiles/datadump.csv

* contains only 3 devices, 3 areas, over 3 dates

* device 1's home is area 1 (appears in area 1 three times, and area 3 once)
* device 2's home is a tie between area 1 and 2 (appears in area 1 and 2 twice)
* device 3 is one of those that only appears once in the sample, still, output should return "home" as that one area 3 it appeared in.

In [3]:
DUMMY_PATH = os.path.join('dummyfiles', 'datadump.csv')
dummy = pd.read_csv(DUMMY_PATH)
dummy

,device_xid,area_id,date
0,1,1,1
1,2,1,1
2,2,2,1
3,1,1,2
4,2,1,2
5,2,2,2
6,1,1,3
7,1,3,3
8,3,3,3
9,2,3,3


In [6]:
dummyhome = (dummy
             .groupby(['device_xid', 'area_id']).size()
             .reset_index()
             .rename(columns={0: 'count'})
             .astype({'count': 'int', 'area_id': 'int', 'device_xid': 'int'})
             .loc[lambda x: x.groupby('device_xid')['count'].transform(max) == x['count']]
             .rename(columns={'area_id': 'home'})
            )

dummyhome

,device_xid,home,count
0,1,1,3
2,2,1,2
3,2,2,2
5,3,3,1


In [7]:
dummyhome.to_csv(os.path.join('dummyfiles', 'dummy-home.csv'), index=False)